# Conversation Q&A Chatbot

in many Q&A applications we want to allow the user to have a back and forth conversation, meaning the application needs some sort of "memory" of past questions and answers, and some logic for incorporating those into its current thinking

in this guide we focus on adding logic for incorporating historical messages. Further details on chat history management is covered in the previous videos

We will cover two approaches:

- Chains, in which we always execute a retrieval step:
- Agents, in which we give an LLM discretion over whether and how to execute a retrieval step (or multiple steps)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm=ChatGroq(model="moonshotai/kimi-k2-instruct-0905" , groq_api_key=groq_api_key)
llm

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain # interface for your vectorDB
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
# 1. Load , chunk and index the contents of the blog to create a retriever

import bs4

loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)


docs = loader.load()
docs

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits = text_splitter.split_documents(docs)

splits

In [ ]:
vectorstore =Chroma.from_documents(documents=splits,embedding=embeddings,persist_directory="./chroma_db")

retriever = vectorstore.as_retriever()

retriever

In [ ]:
## Prompt Template

system_prompt = (
    "You are an assistant for answering questions about the blog post 'Agents: Interactive Decision-Making with Large Language Models' by Lilian Weng. "
    "Use the following retrieved context to answer the question. If you don't know the answer, say you don't know. Always use all available information to answer the question. "
    "Answer the question in a concise and clear manner. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
# chain to combine retrieved documents and prompt template

qa_chain = create_stuff_documents_chain(llm=llm, prompt=prompt) # chain to combine retrieved documents and prompt template maninly used for text summerization where it combines all the retrieved documents into one prompt and then pass it to the llm

rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=qa_chain) # chain to combine retriever and qa_chain

In [ ]:
response  = rag_chain.invoke({"input": "What are the main topics covered in the blog post?"})

response

In [ ]:
rag_chain.invoke({"input": "what is Self-Reflection?"})

In [ ]:
response['answer']

In [ ]:
## Adding Chat History

rag_chain.invoke({"input": "how do we achieve it ?"})['answer'] # inaccurate answer because it doesn't have the chat history

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder # used to include the chat history in the prompt template

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question"
    "which might reference context in the chat history,"
    "formulate a standalone question which can be understood"
    "without the chat history. Do not answer the question,"
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)


In [ ]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt)

history_aware_retriever

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage

chat_history =[]

question = "what is Self-Reflection?"
response = rag_chain.invoke({"input": question, "chat_history": chat_history})

chat_history.extend([HumanMessage(content=question), AIMessage(content=response['answer'])])

questions2= "tell me more about it ?"

response2= rag_chain.invoke({"input": questions2, "chat_history": chat_history})

print(response2['answer'])

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [ ]:
conversational_rag_chain.invoke(
    {"input": "What is Task Decomposition?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

In [ ]:
conversational_rag_chain.invoke(
    {"input": "What are common ways of doing it?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]